In [323]:
import behaviors
import no_signaling_sets
import numpy as np
# from tqdm import tqdm

## Parameters

In [324]:
delta=2
m=2

n_hyperplanes = int(1e5)

## Hyperplane Extraction and Analysis

### One sample plane

In [325]:
non_srns_samples = np.load("../data/non_srns/non_srns_points.npy")
# non_srns_samples = list(non_srns_samples)

example_sample = np.random.choice(len(non_srns_samples), 1)[0]
example_sample = non_srns_samples[example_sample]
example_sample = behaviors.RoutedBehavior(delta, m, vector=example_sample)

srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)

print(f"Example sample: {example_sample}")
print(f"Example sample is no-signaling: [{example_sample.is_no_signaling()}]")

np.linalg.matrix_rank(np.array(non_srns_samples))

Example sample: Behavior:
Short path (z=S):
[[0.248 0.271 0.254 0.486]
 [0.239 0.216 0.325 0.093]
 [0.289 0.354 0.283 0.139]
 [0.224 0.159 0.139 0.282]]
Long path (z=L) :
[[0.423 0.372 0.301 0.071]
 [0.063 0.115 0.277 0.508]
 [0.04  0.027 0.162 0.329]
 [0.474 0.486 0.26  0.093]]
------------
Example sample is no-signaling: [True]


np.int64(15)

### Some rank estimation & stuff

In [326]:
family = non_srns_samples[:n_hyperplanes]
family_rank = np.linalg.matrix_rank(family)
print(f"Rank of family: {family_rank}")

chosen_hyperplane = srns_set.get_facet_hyperplane(example_sample)
hyperplane_behavior = behaviors.RoutedBehavior(delta, m, vector=chosen_hyperplane)

print(f"Hyperplane: {chosen_hyperplane}")
print(f"In behavior expression: {hyperplane_behavior}")

2025-05-26 15:23:36.140 | WARNING  | no_signaling_sets:get_facet_hyperplane:384 - The has rank 11, not yet identified as maximal hyperplane dimension.


Rank of family: 15
Hyperplane: [ 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  1.735  0.     0.     0.     1.735  0.     0.     0.     1.735 -1.735
 -1.735  1.735]
In behavior expression: Behavior:
Short path (z=S):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
Long path (z=L) :
[[ 0.     0.     0.     0.   ]
 [ 1.735  0.     0.     0.   ]
 [ 1.735  0.     0.     0.   ]
 [ 1.735 -1.735 -1.735  1.735]]
------------


## Bulk process samples for all hyperplanes

In [327]:
def scale_down_vector(vector):
    mask = np.abs(vector) > 1e-10

    factor = np.abs(vector[mask])[0]

    abs_is_cst = np.allclose(np.abs(vector[mask]), factor, atol=1e-10)
    # print(f"{np.abs(vector[mask])} == {factor} ? {abs_is_cst}")
    if abs_is_cst:
        # Divide by the factor and round to nearest integer to avoid floating point issues
        rescaled = np.zeros_like(vector)
        rescaled[mask] = np.round(vector[mask] / factor).astype(int)
        return rescaled
    else:
        raise ValueError(f"Vector cannot be scaled down uniformly : {vector}")

def normalize_vector(vector):
    if np.allclose(vector, 0, atol=1e-10):
        return vector
    else:
        return vector / np.linalg.norm(vector)


In [328]:
# all_hyperplanes_scaled = set()
# all_hyperplanes_normalized = set()

# i = 0
# for vec in list(non_srns_samples)[:n_hyperplanes]:
#     _, _, vec_lambda = srns_set.is_facet_hyperplane(
#         behaviors.RoutedBehavior(delta, m, vec)
#         )
#     rescaled_vec = scale_down_vector(vec_lambda)

#     if str(rescaled_vec) not in all_hyperplanes_scaled:
#         all_hyperplanes_scaled.add(str(rescaled_vec))
#         i += 1
#         print(f"[{i}] {behaviors.RoutedBehavior(delta, m, rescaled_vec[:32])}")

# for vec in tqdm(list(non_srns_samples)[:n_hyperplanes]):
#     _, _, vec_lambda = srns_set.is_facet_hyperplane(
#         behaviors.RoutedBehavior(delta, m, vec)
#         )
#     normalized_vec = normalize_vector(vec_lambda)

#     all_hyperplanes_normalized.add(str(normalized_vec))

In [329]:
# import re

# print(f"Number of hyperplanes scaled: {len(all_hyperplanes_scaled)}")
# copy_hyperplanes = list(all_hyperplanes_scaled.copy())
# copy_hyperplanes.sort()
# for hyperplane in copy_hyperplanes:
#     printable = hyperplane.replace("\n", "").replace(".", "")

#     regex_1 = re.sub(r"(\d)\s(\d)", r'\1  \2', printable)
#     regex_2 = re.sub(r"(\d)\s(\d)", r'\1  \2', regex_1)
#     final = re.sub(r"(\[)(\d)", r'\1 \2', regex_2)

#     print(final)

# print(f"Number of hyperplanes normalized: {len(all_hyperplanes_normalized)}")
# copy_hyperplanes_normalized = list(all_hyperplanes_normalized.copy())
# copy_hyperplanes_normalized.sort()
# for hyperplane in copy_hyperplanes_normalized:
#     printable = hyperplane.replace("\n", "")

#     regex_1 = re.sub(r"0\.(\d{7,11})", r'1.', printable)
#     regex_2 = re.sub(r"(\d\.)\s\s+(\d\.)", r'\1 \2', regex_1)
#     regex_2 = re.sub(r"(\d\.)\s\s+(\d\.)", r'\1 \2', regex_2)
#     regex_3 = re.sub(r"(\d)\.\s(\d\.)", r'\1  \2', regex_2)
#     regex_3 = re.sub(r"(\d)\.\s(\d)", r'\1  \2', regex_3)
#     regex_4 = re.sub(r"(\d)\.\s+(\-\d)", r'\1 \2', regex_3)
#     final = re.sub(r"(\d)\s*(\])", r'\1\2', regex_4.replace(".", ""))

#     print(final)

In [330]:
A, b = srns_set.get_equations(example_sample)

with np.printoptions(precision=1, threshold=np.inf):
    for line in A[-4:,1:]:
        print(str(line).replace(".", "").replace("\n", ""))


[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1  0  0]
[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1 -1  0  0  0  0  0  0  1 -1]


## With (2,2,2) hyperplanes in a file, do more work on it

In [331]:
with open("../data/extracted_srns_hyperplanes_comparison.txt", "r") as f:
    lines = f.readlines()

scaled_equations = [line.strip("[]. \n").replace("  ", " ").split(" ") for line in lines[1:33]]
normalized_equations = [line.strip("[]. \n").replace("  "," ").split(" ") for line in lines[34:66]]

print(f"Scaled equations: {len(scaled_equations)}, size {len(scaled_equations[0])}")
print(f"Normalized equations: {len(normalized_equations)}, size {len(normalized_equations[0])}")


Scaled equations: 32, size 36
Normalized equations: 32, size 36


In [332]:
class HashableArray:
    def __init__(self, arr):
        self.array = np.array(arr, dtype=int)

    def __hash__(self):
        return hash(self.array.tobytes())

    def __eq__(self, other):
        if isinstance(other, HashableArray):
            return np.array_equal(self.array, other.array)
        return False

    def __str__(self):
        return str(self.array.__str__())
    
    def __repr__(self):
        return self.__str__()



def format_list_of_hyperplanes(equations: list[list[int]]) -> list[tuple[HashableArray]]:
    formatted: list[tuple[HashableArray]] = []
    for el in equations:
        el_S, el_L, el_NS = el[0:delta**2 * m**2], el[delta**2 * m**2:2 * delta**2 * m**2], el[2 * delta**2 * m**2:]
        el_S = np.array(el_S, dtype=int).reshape((delta, delta, m, m))
        el_L = np.array(el_L, dtype=int).reshape((delta, delta, m, m))
        el_NS = np.array(el_NS, dtype=int).reshape((delta,)*m)

        el_S, el_L, el_NS = HashableArray(el_S), HashableArray(el_L), HashableArray(el_NS)

        formatted.append((el_S, el_L, el_NS))

    return formatted

In [333]:
formatted_scaled = format_list_of_hyperplanes(scaled_equations)
formatted_normal = format_list_of_hyperplanes(normalized_equations)

print(f"Formatted scaled hyperplanes: {formatted_scaled[0]}")
print(f"Formatted normal hyperplanes: {formatted_normal[0]}")

Formatted scaled hyperplanes: ([[[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]


 [[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]], [[[[ 0  0]
   [ 0  0]]

  [[ 0  0]
   [ 0  1]]]


 [[[ 0  0]
   [ 1  0]]

  [[ 1 -1]
   [ 0  0]]]], [[0 1]
 [0 0]])
Formatted normal hyperplanes: ([[[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]


 [[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]], [[[[ 0  0]
   [ 0  0]]

  [[ 0  0]
   [ 1  0]]]


 [[[ 0  0]
   [ 1  0]]

  [[-1  1]
   [ 1 -1]]]], [[0 0]
 [1 0]])


In [334]:
from itertools import permutations


def permute_a(formatted_hyperplane:tuple[HashableArray]) -> list[tuple[HashableArray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of a values."""
    permuted_hyperplanes = []
    for perm in permutations(range(delta)):
        permuted_S = formatted_hyperplane[0].array[list(perm), :, :, :]
        permuted_L = formatted_hyperplane[1].array[list(perm), :, :, :]
        permuted_S, permuted_L = HashableArray(permuted_S), HashableArray(permuted_L)
        permuted_NS = formatted_hyperplane[2]

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

def permute_b(formatted_hyperplane:tuple[HashableArray]) -> list[tuple[HashableArray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of b values."""
    permuted_hyperplanes = []
    for perm in permutations(range(m)):
        permuted_S = formatted_hyperplane[0].array[:, list(perm), :, :]
        permuted_L = formatted_hyperplane[1].array[:, list(perm), :, :]
        permuted_NS = formatted_hyperplane[2].array[np.ix_(*([list(perm)] * m))]
        permuted_S, permuted_L, permuted_NS = HashableArray(permuted_S), HashableArray(permuted_L), HashableArray(permuted_NS)  # noqa: E501

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

def permute_x(formatted_hyperplane:tuple[HashableArray]) -> list[tuple[HashableArray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of x values."""
    permuted_hyperplanes = []
    for perm in permutations(range(m)):
        permuted_S = formatted_hyperplane[0].array[:, :, list(perm), :]
        permuted_L = formatted_hyperplane[1].array[:, :, list(perm), :]
        permuted_S, permuted_L = HashableArray(permuted_S), HashableArray(permuted_L)
        permuted_NS = formatted_hyperplane[2]

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

def permute_y(formatted_hyperplane:tuple[HashableArray]) -> list[tuple[HashableArray]]:
    """Apply to a properly formatted hyperplane the permutation of coordinates
    corresponding to relabelings of y values."""
    permuted_hyperplanes = []
    for perm in permutations(range(m)):
        permuted_S = formatted_hyperplane[0].array[:, :, :, list(perm)]
        permuted_L = formatted_hyperplane[1].array[:, :, :, list(perm)]
        permuted_S, permuted_L = HashableArray(permuted_S), HashableArray(permuted_L)
        permuted_NS = formatted_hyperplane[2]

        permuted_hyperplanes.append((permuted_S, permuted_L, permuted_NS))
    return permuted_hyperplanes

list_of_permutations = [permute_a, permute_b, permute_x, permute_y]

def permute(axis: int|str, formatted_hyperplane: tuple[HashableArray]) -> list[tuple[HashableArray]]:
    """Permute a hyperplane along the specified axis."""
    if isinstance(axis, str):
        axis = axis.lower()
        if axis == "a":
            return permute_a(formatted_hyperplane)
        elif axis == "b":
            return permute_b(formatted_hyperplane)
        elif axis == "x":
            return permute_x(formatted_hyperplane)
        elif axis == "y":
            return permute_y(formatted_hyperplane)
        else:
            raise ValueError(f"Unknown axis: {axis}")
    elif isinstance(axis, int):
        if 0 <= axis < 4:
            return list_of_permutations[axis](formatted_hyperplane)
        else:
            raise ValueError(f"Axis index out of range: {axis}")
    else:
        raise TypeError(f"Axis must be an int or a str, got {type(axis)}")

In [ ]:
def quotient_set(equations: list[tuple[HashableArray]]) -> list[set]:
    """Quotient the set of hyperplanes by the action of the permutation group."""
    quotient = []

    for eq in equations:
        is_represented = False
        for rep in quotient:
            if eq in rep:
                is_represented = True
                # print(f"Hyperplane {eq} is already represented in the quotient set.")
                break

        if not is_represented:
            # print(f"Adding hyperplane {eq} to the quotient set.")
            # We need to represent all permuations of the hyperplane equation,
            # to accurately capture the equivalence class.
            # There are (delta!)^2 * (m!)^2 permutations of the hyperplane.
            equations_set = set()
            equations_set.add(eq)
            for axis in range(4):
                new_equations_set = set()
                for e in equations_set:
                    new_equations_set.update(set(permute(axis, e)))
                equations_set = new_equations_set

            print(len(equations_set))

            quotient.append(equations_set)

    return quotient

In [339]:
scaled_quotient = quotient_set(formatted_scaled)
normal_quotient = quotient_set(formatted_normal)

print(f"Scaled quotient set: {len(scaled_quotient)} equivalence classes")
print(f"Scaled quotient set: {scaled_quotient}")
print()
print(f"Normal quotient set: {len(normal_quotient)} equivalence classes")
print(f"Normal quotient set: {normal_quotient}")
print()
print(f"Comparison: {scaled_quotient == normal_quotient}")

16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
16
Scaled quotient set: 16 equivalence classes
Scaled quotient set: [{([[[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]


 [[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]], [[[[ 0  0]
   [ 0  1]]

  [[ 0  0]
   [ 0  0]]]


 [[[ 1 -1]
   [ 0  0]]

  [[ 0  0]
   [ 1  0]]]], [[0 0]
 [1 0]]), ([[[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]


 [[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]], [[[[ 0  1]
   [ 0  0]]

  [[ 0  0]
   [ 0  0]]]


 [[[ 0  0]
   [ 1 -1]]

  [[ 1  0]
   [ 0  0]]]], [[0 0]
 [1 0]]), ([[[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]


 [[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]], [[[[ 0  0]
   [-1  1]]

  [[ 0  1]
   [ 0  0]]]


 [[[ 1  0]
   [ 0  0]]

  [[ 0  0]
   [ 0  0]]]], [[0 0]
 [1 0]]), ([[[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]


 [[[0 0]
   [0 0]]

  [[0 0]
   [0 0]]]], [[[[ 0  1]
   [ 0  0]]

  [[ 0  0]
   [-1  1]]]


 [[[ 0  0]
   [ 0  0]]

  [[ 1  0]
   [ 0  0]]]], [[0 1]
 [0 0]]), ([[[[0 0]
   [0 0]]

  [[0 0]

In [337]:
my_array = np.arange(2*3*2).reshape((2, 3, 2))
print(f"Shape of my_array: {my_array.shape}")

print(my_array)
print("-"*12)
print(my_array[:, :, [1,0]])
print("-"*12)
print(my_array[:, [2,1,0], :])
print("-"*12)
print(my_array[[1,0], :, :])
print("-"*12)


new_orders = [[1,0], [2,1,0], [1,0]]
reordered = my_array[np.ix_(*new_orders)]
print(reordered)

Shape of my_array: (2, 3, 2)
[[[ 0  1]
  [ 2  3]
  [ 4  5]]

 [[ 6  7]
  [ 8  9]
  [10 11]]]
------------
[[[ 1  0]
  [ 3  2]
  [ 5  4]]

 [[ 7  6]
  [ 9  8]
  [11 10]]]
------------
[[[ 4  5]
  [ 2  3]
  [ 0  1]]

 [[10 11]
  [ 8  9]
  [ 6  7]]]
------------
[[[ 6  7]
  [ 8  9]
  [10 11]]

 [[ 0  1]
  [ 2  3]
  [ 4  5]]]
------------
[[[11 10]
  [ 9  8]
  [ 7  6]]

 [[ 5  4]
  [ 3  2]
  [ 1  0]]]
